In [65]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [66]:
### Token Processing Functions ###

END_PUNCT = {".", "!", "?"}

def tokenize(text: str) -> List[str]:
    """
    - tokenizer
    - The tokens returned includes ending punctuations '.' ',' and '!' 
    - Do not edit this function for pe2.
    """
    pattern = (
        r"(?:[A-Za-z]\.){2,}(?:[A-Za-z]\.)?"   # captures abbreviations like U.S.A.
        r"|[A-Za-z]+(?:['\u2019][A-Za-z]+)*"   # captures words with internal apostrophes
        r"|(?<=[A-Za-z])[.!?]"                 # captures END_PUNCT only at end of a word
    )
    return [t.lower() for t in re.findall(pattern, text)] 

def make_it_pretty(tokens: List[str]) -> str:
    pretty: List[str] = []
    for t in tokens:
        if t in END_PUNCT and pretty:
            pretty[-1] = pretty[-1] + t
        else:
            pretty.append(t)
    if pretty:
        pretty[0] = pretty[0].capitalize()
    return " ".join(pretty)

In [67]:
### Forward Pass Functions ###

def encoding_matrix_generator(T, d, device):
    pe = torch.zeros(T, d).to(device)
    position = torch.arange(0, T, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe.unsqueeze(0)

def RMS_Norm(gamma, X, d, epsilon):
    return gamma * (X / torch.sqrt(1/d * (X*X).sum(dim=-1, keepdim=True) + epsilon) )

def A(X, W_k, W_q, T, d, h, mask):
    return torch.softmax( ( ((X@W_q) @ (X@W_k).mT) / ((d/h)**0.5) ).masked_fill(mask==0, float('-inf')) , dim=-1 )

def MHA(X, A, W_v, W_o):
    return (A @ (X@W_v) @ W_o).sum(dim=-3) #### was org dim=1

def MLP(X, W_1, b_1, W_2, b_2):
    return F.leaky_relu((X@W_1 + b_1.unsqueeze(0)), negative_slope=0.01)@W_2 + b_2.unsqueeze(0)

In [68]:
### Finalizing Vocabulary ###

vocabulary = []

with open('vocab_copy.json', 'r') as file:
    vocabulary = json.load(file)

vocabulary.append("<|endoftext|>")
vocabulary.append("<unk>")
vocabulary.append("<pad>")
vocabulary[-4] = "<sep>"

In [70]:
### Inference ###

choice = input("Enter 1 if you want to generate English stories. Enter 2 if you wan to generate Brainrot: ")

if choice == 1:
    loaded_weights = torch.load("LLM_Weights.pt", weights_only=True)                     # Non-Brainrot weights    
else: 
    loaded_weights = torch.load("LLM_English_to_Brainrot_Weights.pt", weights_only=True) # Brainrot weights


# Scalar Values
layers = 8
v = len(vocabulary)
T = 256
d = 512
h = 8
batch = 16
temp = 0.8
k=10
epsilon = 1e-8

# Weights
embedding_matrix = loaded_weights["embedding_matrix"]
encoding_matrix = encoding_matrix_generator(T, d, device)
W_q = loaded_weights["W_q"]
W_k = loaded_weights["W_k"]
W_v = loaded_weights["W_v"]
W_o = loaded_weights["W_o"]
mask = torch.tril(torch.ones(T, T, device=device))
gamma_MHA = loaded_weights["gamma_MHA"]
W_1 = loaded_weights["W_1"]
b_1 = loaded_weights["b_1"]
W_2 = loaded_weights["W_2"]
b_2 = loaded_weights["b_2"]
gamma_MLP = loaded_weights["gamma_MLP"]
gamma_out = loaded_weights["gamma_out"]
results = torch.rand(T, v).to(device)

# X Value Storing Tensors
X_infer_EmbEnc = torch.zeros(T, d).to(device)
X_infer_postMHA = torch.zeros(layers, T, d).to(device)
X_infer_postMLP = torch.zeros(layers, T, d).to(device)

# Getting input and indicies
sentence_input = input("Enter your text: ")
raw_tokens = tokenize(sentence_input)
original_token_count = len(raw_tokens)
next_token = ""

if choice != 1:
    raw_tokens.append("<sep>")


# Main inference loop
while next_token != "<|endoftext|>" and len(raw_tokens) < 256:
    tokens = []
    
    for i in raw_tokens:
        try: 
            tokens.append( vocabulary.index(i) )
        except ValueError:
            tokens.append( vocabulary.index("<unk>") )
    
    num_tokens = len(tokens)
    
    inference_indicies = torch.tensor(tokens, dtype=torch.long).to(device)
    
    
    X_infer_EmbEnc[ :num_tokens ] = (F.embedding(inference_indicies, embedding_matrix) + encoding_matrix[:, :num_tokens, :]).squeeze(0)
    
    """
    Transformer Layers
    """
    for i in range(layers):
        if i == 0:
            X_infer_postMHA[i][ :num_tokens ] = MHA( RMS_Norm(gamma_MHA[i], X_infer_EmbEnc[:num_tokens], d, epsilon), 
                               A( RMS_Norm(gamma_MHA[i], X_infer_EmbEnc[ :num_tokens ], d, epsilon) , W_k[i], W_q[i], T, d, h, mask[:num_tokens, :num_tokens] ), 
                               W_v[i], W_o[i] ) + X_infer_EmbEnc[ :num_tokens ]
            X_infer_postMLP[i][ :num_tokens ] = MLP( RMS_Norm(gamma_MLP[i], X_infer_postMHA[i][ :num_tokens ], d, epsilon), 
                               W_1[i], b_1[i], W_2[i], b_2[i] ) + X_infer_postMHA[i][ :num_tokens ]
        else:
            X_infer_postMHA[i][ :num_tokens ] = MHA( RMS_Norm(gamma_MHA[i], X_infer_postMLP[i-1][ :num_tokens ], d, epsilon), 
                               A( RMS_Norm(gamma_MHA[i], X_infer_postMLP[i-1][ :num_tokens ], d, epsilon) , W_k[i], W_q[i], T, d, h, mask[:num_tokens, :num_tokens] ), 
                               W_v[i], W_o[i] ) + X_infer_postMLP[i-1][ :num_tokens ]
            X_infer_postMLP[i][ :num_tokens ] = MLP( RMS_Norm(gamma_MLP[i], X_infer_postMHA[i][ :num_tokens ], d, epsilon), 
                               W_1[i], b_1[i], W_2[i], b_2[i] ) + X_infer_postMHA[i][ :num_tokens ]
    
    """
    Decoding
    """
    results[ :num_tokens ] = F.softmax( (RMS_Norm(gamma_out,  X_infer_postMLP[layers-1][ :num_tokens ], d, epsilon) @ embedding_matrix.mT) / temp , dim = -1 )

    #token_id = torch.argmax(results[num_tokens-1], dim=-1).item()

    top_k_logits, top_k_indices = torch.topk(results[num_tokens-1], k=k)
    top_k_probs = F.softmax(top_k_logits, dim=-1)
    sampled_idx = torch.multinomial(top_k_probs, num_samples=1)
    token_id = top_k_indices[sampled_idx].item()
    
    next_token = vocabulary[token_id]
    raw_tokens.append(next_token)
    #print(next_token)


# Determining the LLM output
final_text = []
sep_token = "<sep>"
eot_token = "<|endoftext|>"

try:
    start_idx = raw_tokens.index(sep_token) + 1
    final_text = raw_tokens[start_idx:]
except ValueError:
    final_text = []



if choice == 1:
    try:
        final_text = raw_tokens[original_token_count:-1]
    except ValueError:
        final_text = []
else:
    try:
        start_idx = raw_tokens.index(sep_token) + 1
        final_text = raw_tokens[start_idx:-1]
    except ValueError:
        final_text = []



output_text = make_it_pretty(final_text)

print()
print("LLM Output: " + output_text)

Enter 1 if you want to generate English stories. Enter 2 if you wan to generate Brainrot:  1
Enter your text:  We should find new friends.



LLM Output: She needed something really unique in need to discover new feelings? that is a good wish she explained that
